In [83]:
import pandas as pd
import numpy as np

In [84]:
from nlp_utils import * 

In [85]:
df_train = open_data('./data/liar-plus/train2.tsv')
df_test = open_data('./data/liar-plus/test2.tsv')
df_val = open_data('./data/liar-plus/val2.tsv')
df_train.head()

,id,label,statement,subject,speaker,speaker_job_title,state_info,party_affiliation,barely_true_counts,false_counts,half_true_counts,mostly_true_counts,pants_on_fire_counts,context,justification,party_category,word_count,topic_list
0,2635.json,false,Says the Annies List political group supports ...,abortion,dwayne-bohac,State representative,Texas,republican,0.0,1.0,0.0,0.0,0.0,a mailer,That's a premise that he fails to back up. Ann...,right-leaning,11,[abortion]
1,10540.json,half-true,When did the decline of coal start? It started...,"energy,history,job-accomplishments",scott-surovell,State delegate,Virginia,democrat,0.0,0.0,1.0,1.0,0.0,a floor speech.,"Surovell said the decline of coal ""started whe...",left-leaning,24,"[energy, history, job-accomplishments]"
2,324.json,mostly-true,"Hillary Clinton agrees with John McCain ""by vo...",foreign-policy,barack-obama,President,Illinois,democrat,70.0,71.0,160.0,163.0,9.0,Denver,Obama said he would have voted against the ame...,left-leaning,19,[foreign-policy]
3,1123.json,false,Health care reform legislation is likely to ma...,health-care,blog-posting,NaN,NaN,none,7.0,19.0,3.0,5.0,44.0,a news release,The release may have a point that Mikulskis co...,other,12,[health-care]
4,9028.json,half-true,The economic turnaround started at the end of ...,"economy,jobs",charlie-crist,NaN,Florida,democrat,15.0,9.0,20.0,19.0,2.0,an interview on CNN,"Crist said that the economic ""turnaround start...",left-leaning,10,"[economy, jobs]"


In [86]:
df_train["statement"] = df_train["statement"].astype(str)
df_train.dropna(subset=['statement','label'], inplace=True)
df_train["statement"].apply(clean_text)
df_train = df_train[['statement','label']]

df_val["statement"] = df_val["statement"].astype(str)
df_val.dropna(subset=['statement','label'], inplace=True)
df_val["statement"].apply(clean_text)
df_val = df_val[['statement','label']]

df_test["statement"] = df_test["statement"].astype(str)
df_test.dropna(subset=['statement','label'], inplace=True)
df_test["statement"].apply(clean_text)
df_test = df_test[['statement','label']]

In [87]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sentence_transformers import SentenceTransformer
from torch.utils.data import DataLoader, TensorDataset

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No CUDA GPU")

2.9.0+cu126
True
NVIDIA GeForce RTX 3060


# Old

In [ ]:
from sentence_transformers import SentenceTransformer

# model_st = SentenceTransformer('qwen/Qwen3-Embedding-0.6B')
# model_st = SentenceTransformer('all-miniLM-L6-v2')
model_st = SentenceTransformer('BAAI/bge-large-en-v1.5', device='cuda')

for df in [df_train, df_test, df_val]:
    embeddings = model_st.encode(df['statement'].tolist(), batch_size=64, convert_to_tensor=True, show_progress_bar=True)
    df['embeddings'] = embeddings #.tolist()

Batches:   0%|          | 0/160 [00:00<?, ?it/s]

Batches:   0%|          | 0/20 [00:00<?, ?it/s]

Batches:   0%|          | 0/21 [00:00<?, ?it/s]

In [39]:
df_train

,id,label,statement,subject,speaker,speaker_job_title,state_info,party_affiliation,barely_true_counts,false_counts,half_true_counts,mostly_true_counts,pants_on_fire_counts,context,justification,party_category,word_count,topic_list,embeddings
0,2635.json,false,Says the Annies List political group supports ...,abortion,dwayne-bohac,State representative,Texas,republican,0.0,1.0,0.0,0.0,0.0,a mailer,That's a premise that he fails to back up. Ann...,right-leaning,11,[abortion],"[[tensor(-0.6520, device='cuda:0'), tensor(0.5..."
1,10540.json,half-true,When did the decline of coal start? It started...,"energy,history,job-accomplishments",scott-surovell,State delegate,Virginia,democrat,0.0,0.0,1.0,1.0,0.0,a floor speech.,"Surovell said the decline of coal ""started whe...",left-leaning,24,"[energy, history, job-accomplishments]","[[tensor(0.0830, device='cuda:0'), tensor(-0.3..."
2,324.json,mostly-true,"Hillary Clinton agrees with John McCain ""by vo...",foreign-policy,barack-obama,President,Illinois,democrat,70.0,71.0,160.0,163.0,9.0,Denver,Obama said he would have voted against the ame...,left-leaning,19,[foreign-policy],"[[tensor(0.3764, device='cuda:0'), tensor(-0.4..."
3,1123.json,false,Health care reform legislation is likely to ma...,health-care,blog-posting,NaN,NaN,none,7.0,19.0,3.0,5.0,44.0,a news release,The release may have a point that Mikulskis co...,other,12,[health-care],"[[tensor(-0.1530, device='cuda:0'), tensor(-0...."
4,9028.json,half-true,The economic turnaround started at the end of ...,"economy,jobs",charlie-crist,NaN,Florida,democrat,15.0,9.0,20.0,19.0,2.0,an interview on CNN,"Crist said that the economic ""turnaround start...",left-leaning,10,"[economy, jobs]","[[tensor(-0.4501, device='cuda:0'), tensor(-0...."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10237,5473.json,mostly-true,There are a larger number of shark attacks in ...,"animals,elections",aclu-florida,NaN,Florida,none,0.0,1.0,1.0,1.0,0.0,"interview on ""The Colbert Report""",They compounded their error by combining full ...,other,17,"[animals, elections]","[[tensor(0.1031, device='cuda:0'), tensor(0.81..."
10238,3408.json,mostly-true,Democrats have now become the party of the [At...,elections,alan-powell,NaN,Georgia,republican,0.0,0.0,0.0,1.0,0.0,an interview,"Romney said that ""Obamacare means that for up...",right-leaning,14,[elections],"[[tensor(0.1859, device='cuda:0'), tensor(0.71..."
10239,3959.json,half-true,Says an alternative to Social Security that op...,"retirement,social-security",herman-cain,NaN,Georgia,republican,4.0,11.0,5.0,3.0,3.0,a Republican presidential debate,But that it leaves out important details and t...,right-leaning,28,"[retirement, social-security]","[[tensor(-0.4729, device='cuda:0'), tensor(0.7..."
10240,2253.json,false,On lifting the U.S. Cuban embargo and allowing...,"florida,foreign-policy",jeff-greene,NaN,Florida,democrat,3.0,1.0,3.0,0.0,0.0,a televised debate on Miami's WPLG-10 against ...,"We checked the research and, quite frankly, fi...",left-leaning,11,"[florida, foreign-policy]","[[tensor(-0.0157, device='cuda:0'), tensor(0.2..."


In [ ]:
le = LabelEncoder()
X_train = np.array(df_train['embeddings'].tolist())
y_train = le.fit_transform(df_train['label'])
X_test = np.array(df_test['embeddings'].tolist())
y_test = le.transform(df_test['label'])
X_val = np.array(df_val['embeddings'].tolist())
y_val = le.transform(df_val['label'])

In [44]:
import torch.nn.functional as F

# X_train = torch.tensor(X_train, dtype=torch.float32)
# y_train = torch.tensor(y_train, dtype=torch.long)
# X_test = torch.tensor(X_test, dtype=torch.float32)
# y_test = torch.tensor(y_test, dtype=torch.long)
# X_val = torch.tensor(X_val, dtype=torch.float32)
# y_val = torch.tensor(y_val, dtype=torch.long)

# normalize
X_train = F.normalize(X_train, p=2, dim=1)
X_val   = F.normalize(X_val, p=2, dim=1)
X_test  = F.normalize(X_test, p=2, dim=1)

train_data = TensorDataset(X_train, y_train)
test_data = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64)

val_data = TensorDataset(X_val, y_val)
val_loader = DataLoader(val_data, batch_size=64)

AttributeError: 'Series' object has no attribute 'norm'

In [ ]:
activation_fn = nn.LeakyReLU()

class LiarClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super(LiarClassifier, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            activation_fn,
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, hidden_dim//2),
            activation_fn,
            nn.Linear(hidden_dim//2, hidden_dim//4),
            activation_fn,
            nn.Linear(hidden_dim//4, num_classes)
        )

    def forward(self, x):
        return self.net(x)

class LiarCNN(nn.Module):
    def __init__(self, num_classes=6):
        super(LiarCNN, self).__init__()
        
        self.conv1 = nn.Conv1d(1, 64, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(64, 128, kernel_size=5, padding=2)
        self.conv3 = nn.Conv1d(128, 256, kernel_size=7, padding=3)
        
        self.bn1 = nn.BatchNorm1d(64)
        self.bn2 = nn.BatchNorm1d(128)
        self.bn3 = nn.BatchNorm1d(256)
        
        self.global_pool = nn.AdaptiveMaxPool1d(1)
        self.fc1 = nn.Linear(256, 128)
        self.dropout = nn.Dropout(0.4)
        self.fc2 = nn.Linear(128, num_classes)
        
    def forward(self, x):
        x = x.unsqueeze(1)
        
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.relu(self.bn3(self.conv3(x)))
        
        x = self.global_pool(x).squeeze(-1)
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.fc2(x)
        return x
    


input_dim = X_train.shape[1]
hidden_dim = 512
num_classes = len(le.classes_)

model = LiarClassifier(input_dim, hidden_dim, num_classes)
# model = LiarCNN(input_dim, num_classes)
model.to(device)

LiarCNN(
  (conv1): Conv1d(1, 64, kernel_size=(3,), stride=(1,), padding=(1,))
  (conv2): Conv1d(64, 128, kernel_size=(5,), stride=(1,), padding=(2,))
  (conv3): Conv1d(128, 256, kernel_size=(7,), stride=(1,), padding=(3,))
  (bn1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (bn2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (bn3): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (global_pool): AdaptiveMaxPool1d(output_size=1)
  (fc1): Linear(in_features=256, out_features=128, bias=True)
  (dropout): Dropout(p=0.4, inplace=False)
  (fc2): Linear(in_features=128, out_features=6, bias=True)
)

In [ ]:
from sklearn.metrics import accuracy_score

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)
epochs = 50

for epoch in range(epochs):
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_train_loss = total_loss / len(train_loader)

    model.eval()
    val_loss = 0
    y_true, y_pred = [], []
    
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            val_loss += loss.item()
            
            preds = torch.argmax(outputs, dim=1)
            y_pred.extend(preds.cpu().tolist())
            y_true.extend(y_batch.cpu().tolist())
    
    avg_val_loss = val_loss / len(val_loader)
    val_acc = accuracy_score(y_true, y_pred)
    
    print(f"Epoch {epoch+1}/{epochs} | "
          f"Train Loss: {avg_train_loss:.4f} | "
          f"Val Loss: {avg_val_loss:.4f} | "
          f"Val Acc: {val_acc:.4f}")


Epoch 1/50 | Train Loss: 1.8240 | Val Loss: 1.7636 | Val Acc: 0.1931
Epoch 2/50 | Train Loss: 1.7577 | Val Loss: 1.7648 | Val Acc: 0.1931
Epoch 3/50 | Train Loss: 1.7582 | Val Loss: 1.7606 | Val Acc: 0.1931
Epoch 4/50 | Train Loss: 1.7585 | Val Loss: 1.7623 | Val Acc: 0.1931
Epoch 5/50 | Train Loss: 1.7579 | Val Loss: 1.7626 | Val Acc: 0.1931
Epoch 6/50 | Train Loss: 1.7584 | Val Loss: 1.7625 | Val Acc: 0.1931
Epoch 7/50 | Train Loss: 1.7575 | Val Loss: 1.7648 | Val Acc: 0.1931
Epoch 8/50 | Train Loss: 1.7580 | Val Loss: 1.7613 | Val Acc: 0.1931
Epoch 9/50 | Train Loss: 1.7579 | Val Loss: 1.7651 | Val Acc: 0.1931
Epoch 10/50 | Train Loss: 1.7581 | Val Loss: 1.7641 | Val Acc: 0.1931
Epoch 11/50 | Train Loss: 1.7583 | Val Loss: 1.7629 | Val Acc: 0.1931
Epoch 12/50 | Train Loss: 1.7584 | Val Loss: 1.7624 | Val Acc: 0.1931
Epoch 13/50 | Train Loss: 1.7576 | Val Loss: 1.7627 | Val Acc: 0.1931
Epoch 14/50 | Train Loss: 1.7577 | Val Loss: 1.7679 | Val Acc: 0.2048
Epoch 15/50 | Train Loss: 1.7

KeyboardInterrupt: 

In [54]:
from sklearn.metrics import accuracy_score, classification_report

model.eval()
y_pred, y_true = [], []

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model.to(device)
# model.eval()
y_pred, y_true = [], []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        # Move both input and labels to the same device as model
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        outputs = model(X_batch)
        preds = torch.argmax(outputs, dim=1)

        # Move predictions and labels back to CPU for sklearn
        y_pred.extend(preds.cpu().tolist())
        y_true.extend(y_batch.cpu().tolist())

print("Accuracy:", accuracy_score(y_true, y_pred))
print(classification_report(y_true, y_pred, target_names=le.classes_))

Accuracy: 0.24546172059984214
              precision    recall  f1-score   support

 barely-true       0.21      0.16      0.18       212
       false       0.25      0.31      0.27       249
   half-true       0.25      0.27      0.26       265
 mostly-true       0.23      0.24      0.23       241
  pants-fire       0.23      0.17      0.20        92
        true       0.31      0.27      0.29       208

    accuracy                           0.25      1267
   macro avg       0.24      0.24      0.24      1267
weighted avg       0.25      0.25      0.24      1267



# New

In [88]:
# -------------------------
# Imports
# -------------------------
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
import pandas as pd

# -------------------------
# Step 1. Clean data
# -------------------------
for df in [df_train, df_val, df_test]:
    df.dropna(subset=["statement", "label"], inplace=True)

le = LabelEncoder()
df_train["label_enc"] = le.fit_transform(df_train["label"])
df_val["label_enc"]   = le.transform(df_val["label"])
df_test["label_enc"]  = le.transform(df_test["label"])

# -------------------------
# Step 2. Embedding
# -------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
model_st = SentenceTransformer("all-mpnet-base-v2", device=device)
# model_st = SentenceTransformer("facebook/bart-large-mnli", device=device)

X_train = model_st.encode(df_train["statement"].tolist(), convert_to_tensor=True, show_progress_bar=True)
X_val   = model_st.encode(df_val["statement"].tolist(), convert_to_tensor=True, show_progress_bar=True)
X_test  = model_st.encode(df_test["statement"].tolist(), convert_to_tensor=True, show_progress_bar=True)

# Normalize embeddings (unit length)
X_train = F.normalize(X_train, p=2, dim=1)
X_val   = F.normalize(X_val, p=2, dim=1)
X_test  = F.normalize(X_test, p=2, dim=1)

# Standardize embeddings
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train = torch.tensor(scaler.fit_transform(X_train.cpu().numpy())).to(device)
X_val =  torch.tensor(scaler.transform(X_val.cpu().numpy())).to(device)
X_test =  torch.tensor(scaler.transform(X_test.cpu().numpy())).to(device)

# -------------------------
# Step 3. Convert labels
# -------------------------
y_train = torch.tensor(df_train["label_enc"].astype(int).values, dtype=torch.long, device=device)
y_val   = torch.tensor(df_val["label_enc"].astype(int).values, dtype=torch.long, device=device)
y_test  = torch.tensor(df_test["label_enc"].astype(int).values, dtype=torch.long, device=device)

# -------------------------
# Step 4. DataLoaders
# -------------------------
train_data = TensorDataset(X_train, y_train)
val_data   = TensorDataset(X_val, y_val)
test_data  = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_data, batch_size=64)
test_loader  = DataLoader(test_data, batch_size=64)


Batches:   0%|          | 0/320 [00:00<?, ?it/s]

Batches:   0%|          | 0/41 [00:00<?, ?it/s]

Batches:   0%|          | 0/40 [00:00<?, ?it/s]

In [89]:
import numpy as np

# Model with stronger regularization
class LiarMLP(nn.Module):
    def __init__(self, input_dim=384, hidden_dim=256, num_classes=6):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.5),           # ↑ dropout
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.5),           # add another dropout layer
            nn.Linear(hidden_dim // 2, num_classes)
        )

    def forward(self, x):
        return self.net(x)

class LiarResidualMLP(nn.Module):
    def __init__(self, input_dim=768, hidden_dim=512, num_classes=6):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, num_classes)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(0.4)

    def forward(self, x):
        h = F.relu(self.norm(self.fc1(x)))
        h = self.dropout(h)
        h2 = F.relu(self.norm(self.fc2(h))) + h  # residual connection
        h2 = self.dropout(h2)
        return self.fc3(h2)
    
    def embed(self, x):
        # Return the hidden representation before the final output
        h = F.relu(self.norm(self.fc1(x)))
        h = F.relu(self.norm(self.fc2(h))) + h  # residual
        return h

model = LiarResidualMLP(input_dim=X_train.shape[1], num_classes=len(le.classes_)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=5e-5, weight_decay=1e-4)  # smaller LR, stronger L2
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

epochs = 200
best_val_loss = np.inf
patience, counter = 5, 0  # early stopping patience

for epoch in range(epochs):
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    # Validation
    model.eval()
    val_loss, y_true, y_pred = 0, [], []
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            val_loss += loss.item()
            preds = torch.argmax(outputs, dim=1)
            y_pred.extend(preds.cpu().tolist())
            y_true.extend(y_batch.cpu().tolist())

    avg_train_loss = total_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)
    val_acc = accuracy_score(y_true, y_pred)
    scheduler.step(avg_val_loss)

    print(f"Epoch {epoch+1}/{epochs} | Train Loss: {avg_train_loss:.4f} | "
          f"Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.4f}")

    # Early stopping
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        counter = 0
        best_state = model.state_dict()
    else:
        counter += 1
        if counter >= patience:
            print("Early stopping triggered!")
            break

model.load_state_dict(best_state)


Epoch 1/200 | Train Loss: 1.9502 | Val Loss: 1.7407 | Val Acc: 0.2360
Epoch 2/200 | Train Loss: 1.8357 | Val Loss: 1.7106 | Val Acc: 0.2671
Epoch 3/200 | Train Loss: 1.7711 | Val Loss: 1.7011 | Val Acc: 0.2710
Epoch 4/200 | Train Loss: 1.7352 | Val Loss: 1.6892 | Val Acc: 0.2788
Epoch 5/200 | Train Loss: 1.6957 | Val Loss: 1.6865 | Val Acc: 0.2718
Epoch 6/200 | Train Loss: 1.6631 | Val Loss: 1.6816 | Val Acc: 0.2812
Epoch 7/200 | Train Loss: 1.6375 | Val Loss: 1.6816 | Val Acc: 0.2780
Epoch 8/200 | Train Loss: 1.6093 | Val Loss: 1.6832 | Val Acc: 0.2796
Epoch 9/200 | Train Loss: 1.5819 | Val Loss: 1.6826 | Val Acc: 0.2882
Epoch 10/200 | Train Loss: 1.5580 | Val Loss: 1.6831 | Val Acc: 0.2835
Epoch 11/200 | Train Loss: 1.5434 | Val Loss: 1.6840 | Val Acc: 0.2765
Epoch 12/200 | Train Loss: 1.5349 | Val Loss: 1.6832 | Val Acc: 0.2804
Early stopping triggered!


<All keys matched successfully>

In [90]:
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import f1_score

model.eval()
y_pred, y_true = [], []

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model.to(device)
# model.eval()
y_pred, y_true = [], []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        # Move both input and labels to the same device as model
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        outputs = model(X_batch)
        preds = torch.argmax(outputs, dim=1)

        # Move predictions and labels back to CPU for sklearn
        y_pred.extend(preds.cpu().tolist())
        y_true.extend(y_batch.cpu().tolist())

f1_macro = f1_score(y_true, y_pred, average='macro')
print("Accuracy:", accuracy_score(y_true, y_pred))
print("F1 Macro:", f1_macro)
print(classification_report(y_true, y_pred, target_names=le.classes_))

Accuracy: 0.2809786898184688
F1 Macro: 0.2758018714996346
              precision    recall  f1-score   support

 barely-true       0.28      0.14      0.19       212
       false       0.26      0.34      0.30       249
   half-true       0.29      0.29      0.29       265
 mostly-true       0.25      0.31      0.28       241
  pants-fire       0.33      0.24      0.28        92
        true       0.33      0.33      0.33       208

    accuracy                           0.28      1267
   macro avg       0.29      0.27      0.28      1267
weighted avg       0.28      0.28      0.28      1267



In [91]:
model.eval()
with torch.no_grad():
    # -------------------
    # Train
    # -------------------
    df_train_features = model_st.encode(df_train["statement"].tolist(), convert_to_tensor=True, device=device)
    train_feats = model.embed(df_train_features).cpu().numpy()
    train_cols = [f"mlp_emb_{i}" for i in range(train_feats.shape[1])]
    df_train = pd.concat([df_train.reset_index(drop=True), pd.DataFrame(train_feats, columns=train_cols)], axis=1)

    # -------------------
    # Validation
    # -------------------
    df_val_features = model_st.encode(df_val["statement"].tolist(), convert_to_tensor=True, device=device)
    val_feats = model.embed(df_val_features).cpu().numpy()
    val_cols = [f"mlp_emb_{i}" for i in range(val_feats.shape[1])]
    df_val = pd.concat([df_val.reset_index(drop=True), pd.DataFrame(val_feats, columns=val_cols)], axis=1)

    # -------------------
    # Test
    # -------------------
    df_test_features = model_st.encode(df_test["statement"].tolist(), convert_to_tensor=True, device=device)
    test_feats = model.embed(df_test_features).cpu().numpy()
    test_cols = [f"mlp_emb_{i}" for i in range(test_feats.shape[1])]
    df_test = pd.concat([df_test.reset_index(drop=True), pd.DataFrame(test_feats, columns=test_cols)], axis=1)


# Factuality 2: Spam

In [92]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import pandas as pd

spam_model_name = "mrm8488/bert-tiny-finetuned-sms-spam-detection"
spam_tokenizer = AutoTokenizer.from_pretrained(spam_model_name)
spam_model = AutoModelForSequenceClassification.from_pretrained(spam_model_name)
spam_model.eval()

def get_spam_scores(text_list, batch_size=16):
    scores = []
    for i in range(0, len(text_list), batch_size):
        batch = text_list[i:i+batch_size]
        inputs = spam_tokenizer(
            batch,
            return_tensors="pt",
            truncation=True,
            padding="max_length",
            max_length=512
        )
        with torch.no_grad():
            outputs = spam_model(**inputs)
            probs = torch.softmax(outputs.logits, dim=1)
            scores.extend(probs[:, 1].tolist())
    return scores

df_train['spam_score'] = get_spam_scores(df_train['statement'].tolist())
df_test['spam_score'] = get_spam_scores(df_test['statement'].tolist())
df_val['spam_score'] = get_spam_scores(df_val['statement'].tolist())

# Factuality 3: Political Bias

In [93]:
import spacy
# spacy.cli.download("en_core_web_md")
datum = df_train.iloc[0]
nlp = spacy.load("en_core_web_md")
doc = nlp(datum['statement'])
doc.vector.shape
statistic_types = {'CARDINAL', 'PERCENT', 'MONEY', 'QUANTITY'}

def stat_counter(text):
    if not isinstance(text, str):
        return 0
    doc = nlp(text)
    counter = 0
    for ent in doc.ents: 
        if ent.label_ in statistic_types:
            counter += 1
    return counter

from rapidfuzz import fuzz
conservative_bigrams = pd.read_csv('top_conservative_bigrams.csv')['bigram']
liberal_bigrams = pd.read_csv('top_liberal_bigrams.csv')['bigram']
def match_counter(statement, bigram_list, threshold):
    stat = nlp(str(statement))
    word = [word.text.lower() for word in stat]
    bigram_coll = [''.join(word[i:i+2]) for i in range(len(word)-1)]
    matches = 0
    for bigram in bigram_coll:
        for check in bigram_list:
            if fuzz.ratio(bigram, check) >= threshold:
                matches += 1
                break

    return matches


df_train['statistic_count'] = df_train['statement'].apply(stat_counter)
df_train['conservative_bigram_count'] = df_train['statement'].apply(
    lambda x: match_counter(x, conservative_bigrams, threshold=70)
)
df_train['liberal_bigram_count'] = df_train['statement'].apply(
    lambda x: match_counter(x, liberal_bigrams, threshold=70)
)

df_val['statistic_count'] = df_val['statement'].apply(stat_counter)
df_val['conservative_bigram_count'] = df_val['statement'].apply(
    lambda x: match_counter(x, conservative_bigrams, threshold=70)
)
df_val['liberal_bigram_count'] = df_val['statement'].apply(
    lambda x: match_counter(x, liberal_bigrams, threshold=70)
)

df_test['statistic_count'] = df_test['statement'].apply(stat_counter)
df_test['conservative_bigram_count'] = df_test['statement'].apply(
    lambda x: match_counter(x, conservative_bigrams, threshold=70)
)
df_test['liberal_bigram_count'] = df_test['statement'].apply(
    lambda x: match_counter(x, liberal_bigrams, threshold=70)
)

# Factuality 4: Sensationalism

In [94]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

def emotional_intensity_vader(text):
    if not isinstance(text, str):
        return 0.0
    if len(text) == 0:
        return 0.0
    vs = analyzer.polarity_scores(text)
    return abs(vs['compound'])

for df in [df_train, df_test, df_val]:
    df['emotional_intensity'] = df['statement'].apply(emotional_intensity_vader)

# Statement Embedding

In [95]:
sentence_model = SentenceTransformer("all-MiniLM-L6-v2", device=device)


def embed_statement(text, chunk_size=200, overlap=50):
    words = str(text).split()
    if len(words) <= chunk_size:
        return sentence_model.encode([text])[0]
    chunks = [" ".join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size - overlap)]
    embeddings = sentence_model.encode(chunks)
    return np.mean(embeddings, axis=0) 

df_train["embedding"] = df_train["statement"].apply(embed_statement)
df_test["embedding"]  = df_test["statement"].apply(embed_statement)
df_val["embedding"]   = df_val["statement"].apply(embed_statement)

train_embeddings = np.vstack(df_train["embedding"].values)
train_cols = [f"emb_{i}" for i in range(train_embeddings.shape[1])]
df_train = pd.concat([df_train.reset_index(drop=True), pd.DataFrame(train_embeddings, columns=train_cols)], axis=1)
df_train.drop(columns=["embedding"], inplace=True)

val_embeddings = np.vstack(df_val["embedding"].values)
val_cols = [f"emb_{i}" for i in range(val_embeddings.shape[1])]
df_val = pd.concat([df_val.reset_index(drop=True), pd.DataFrame(val_embeddings, columns=val_cols)], axis=1)
df_val.drop(columns=["embedding"], inplace=True)

test_embeddings = np.vstack(df_test["embedding"].values)
test_cols = [f"emb_{i}" for i in range(test_embeddings.shape[1])]
df_test = pd.concat([df_test.reset_index(drop=True), pd.DataFrame(test_embeddings, columns=test_cols)], axis=1)
df_test.drop(columns=["embedding"], inplace=True)


# Muller Loop

In [96]:
df_train

,statement,label,label_enc,mlp_emb_0,mlp_emb_1,mlp_emb_2,mlp_emb_3,mlp_emb_4,mlp_emb_5,mlp_emb_6,...,emb_374,emb_375,emb_376,emb_377,emb_378,emb_379,emb_380,emb_381,emb_382,emb_383
0,Says the Annies List political group supports ...,false,1,0.253486,0.000000,1.175326,0.824225,10.822678,0.112347,1.603347,...,0.061642,0.031746,-0.080949,-0.010309,0.062963,0.025709,0.084539,-0.060905,0.166828,0.039760
1,When did the decline of coal start? It started...,half-true,2,0.000000,0.409667,0.000000,0.673277,9.510511,0.595865,0.833258,...,0.061461,-0.002610,0.044167,-0.001004,-0.078507,-0.079317,-0.010799,0.017237,-0.061281,-0.001477
2,"Hillary Clinton agrees with John McCain ""by vo...",mostly-true,3,0.000000,0.043899,0.648915,0.130534,10.814679,1.453092,0.235415,...,0.037187,0.086657,-0.045857,-0.075116,-0.039440,0.003276,0.005033,0.050504,0.071088,-0.070123
3,Health care reform legislation is likely to ma...,false,1,0.000000,0.668484,1.456415,1.301655,10.619506,0.073639,0.197742,...,0.002507,0.070096,0.038114,-0.030645,-0.006283,0.054631,0.021744,-0.128516,0.086446,-0.095068
4,The economic turnaround started at the end of ...,half-true,2,0.000000,1.439671,0.530698,0.000000,10.597538,0.137506,0.000000,...,0.099117,-0.077822,0.015376,0.015565,-0.135522,0.028486,-0.001721,-0.096078,-0.027198,0.095355
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10235,There are a larger number of shark attacks in ...,mostly-true,3,0.000000,0.000000,0.000000,0.255663,10.519302,2.277553,0.753438,...,-0.013047,0.027880,0.039358,-0.039163,0.040587,-0.012937,0.055977,-0.027015,0.017876,0.047906
10236,Democrats have now become the party of the [At...,mostly-true,3,0.000000,0.200126,1.196772,0.412083,11.551872,0.895733,0.517323,...,0.039730,-0.128908,-0.005140,0.023496,0.041325,-0.030298,0.016746,0.009866,0.040038,-0.015803
10237,Says an alternative to Social Security that op...,half-true,2,0.000000,0.000000,1.213423,0.000000,10.982914,0.572050,0.000000,...,-0.050827,-0.023327,0.051421,0.041582,-0.057477,-0.028695,0.024356,-0.076488,0.029235,0.088331
10238,On lifting the U.S. Cuban embargo and allowing...,false,1,0.000000,1.064189,0.335229,0.000000,9.965334,1.641799,0.128158,...,0.070683,0.067400,-0.001681,0.003638,0.058002,-0.008409,0.055646,-0.077675,0.003842,-0.037684


In [110]:
X_train = df_train.drop(columns = ['statement','label','label_enc'])
X_val = df_val.drop(columns = ['statement','label','label_enc'])
X_test = df_test.drop(columns = ['statement','label', 'label_enc'])

y_train = df_train["label_enc"].astype(int).values
y_val = df_val["label_enc"].astype(int).values
y_test = df_test["label_enc"].astype(int).values

In [98]:
from time import time
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_moons, make_circles, make_classification
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis

names = ["Nearest Neighbors", "Linear SVM", "RBF SVM", #"Gaussian Process",
         "Decision Tree", "Random Forest", "Neural Net", "AdaBoost",
         "Naive Bayes", "QDA"]

classifiers = [
    KNeighborsClassifier(2),
    SVC(kernel="linear", C=0.025),
    SVC(gamma=2, C=1),
#     GaussianProcessClassifier(1.0 * RBF(1.0)),
    DecisionTreeClassifier(max_depth=5),
    RandomForestClassifier(max_depth=5, n_estimators=10, max_features=1),
    MLPClassifier(alpha=1, max_iter=1000),
    AdaBoostClassifier(),
    GaussianNB(),
    QuadraticDiscriminantAnalysis()]

# TODO (Apply): All cross-validation

max_score = 0.0
max_class = ''
# iterate over classifiers
for name, clf in zip(names, classifiers):
    start_time = time()
    clf.fit(X_train, y_train)
    score = 100.0 * clf.score(X_test, y_test)
    print('Classifier = %s, Score (test, accuracy) = %.2f,' %(name, score), 'Training time = %.2f seconds' % (time() - start_time))
    
    if score > max_score:
        clf_best = clf
        max_score = score
        max_class = name

print(80*'-' )
print('Best --> Classifier = %s, Score (test, accuracy) = %.2f' %(max_class, max_score))

Classifier = Nearest Neighbors, Score (test, accuracy) = 22.26, Training time = 0.44 seconds
Classifier = Linear SVM, Score (test, accuracy) = 24.47, Training time = 56.86 seconds
Classifier = RBF SVM, Score (test, accuracy) = 21.15, Training time = 82.27 seconds
Classifier = Decision Tree, Score (test, accuracy) = 21.78, Training time = 4.82 seconds
Classifier = Random Forest, Score (test, accuracy) = 20.99, Training time = 0.10 seconds
Classifier = Neural Net, Score (test, accuracy) = 24.47, Training time = 48.38 seconds


c:\Users\Chris Mo\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(


Classifier = AdaBoost, Score (test, accuracy) = 23.20, Training time = 54.79 seconds
Classifier = Naive Bayes, Score (test, accuracy) = 24.86, Training time = 0.25 seconds


c:\Users\Chris Mo\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\discriminant_analysis.py:949: UserWarning: Variables are collinear
  warnings.warn("Variables are collinear")


Classifier = QDA, Score (test, accuracy) = 23.13, Training time = 5.67 seconds
--------------------------------------------------------------------------------
Best --> Classifier = Naive Bayes, Score (test, accuracy) = 24.86


In [105]:
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

# -------------------------
# Prepare DMatrix
# -------------------------
dtrain = xgb.DMatrix(X_train, label=y_train)
dval = xgb.DMatrix(X_val, label=y_val)
dtest = xgb.DMatrix(X_test, label=y_test)

num_classes = len(np.unique(y_train))

params = {
    "objective": "multi:softmax",
    "num_class": num_classes,
    "eval_metric": "mlogloss",
    "eta": 0.05,
    "max_depth": 6,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "seed": 42
}

evallist = [(dtrain, "train"), (dval, "eval")]

# -------------------------
# Train with early stopping
# -------------------------
bst = xgb.train(
    params,
    dtrain,
    num_boost_round=500,
    evals=evallist,
    early_stopping_rounds=20,
    verbose_eval=True
)

# -------------------------
# Predictions
# -------------------------
y_pred_val = bst.predict(dval)
y_pred_test = bst.predict(dtest)

print("Validation Accuracy:", accuracy_score(y_val, y_pred_val))
print("Validation Classification Report:\n", classification_report(y_val, y_pred_val))
print("Test Accuracy:", accuracy_score(y_test, y_pred_test))
print("Test Classification Report:\n", classification_report(y_test, y_pred_test))


[0]	train-mlogloss:1.77121	eval-mlogloss:1.78677
[1]	train-mlogloss:1.75143	eval-mlogloss:1.78260
[2]	train-mlogloss:1.73277	eval-mlogloss:1.77824
[3]	train-mlogloss:1.71359	eval-mlogloss:1.77379
[4]	train-mlogloss:1.69575	eval-mlogloss:1.76950
[5]	train-mlogloss:1.67800	eval-mlogloss:1.76594
[6]	train-mlogloss:1.66067	eval-mlogloss:1.76212
[7]	train-mlogloss:1.64417	eval-mlogloss:1.75754
[8]	train-mlogloss:1.62771	eval-mlogloss:1.75461
[9]	train-mlogloss:1.61143	eval-mlogloss:1.74994
[10]	train-mlogloss:1.59570	eval-mlogloss:1.74594
[11]	train-mlogloss:1.58121	eval-mlogloss:1.74314
[12]	train-mlogloss:1.56601	eval-mlogloss:1.74082
[13]	train-mlogloss:1.55148	eval-mlogloss:1.73813
[14]	train-mlogloss:1.53695	eval-mlogloss:1.73544
[15]	train-mlogloss:1.52249	eval-mlogloss:1.73336
[16]	train-mlogloss:1.50829	eval-mlogloss:1.73123
[17]	train-mlogloss:1.49454	eval-mlogloss:1.72841
[18]	train-mlogloss:1.48046	eval-mlogloss:1.72607
[19]	train-mlogloss:1.46726	eval-mlogloss:1.72416
[20]	train

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler


X_train = torch.tensor(X_train.values, dtype=torch.float32).to(device)
y_train = torch.tensor(y_train, dtype=torch.long).to(device)

X_val = torch.tensor(X_val.values, dtype=torch.float32)
y_val = torch.tensor(y_val, dtype=torch.long)

X_test = torch.tensor(X_test.values, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.long)

train_data = TensorDataset(X_train, y_train)
val_data = TensorDataset(X_val, y_val)
test_data = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=64)
test_loader = DataLoader(test_data, batch_size=64)


TypeError: must be real number, not builtin_function_or_method

In [ ]:
class FullFeatureCNN(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(FullFeatureCNN, self).__init__()
        self.conv1 = nn.Conv1d(1, 128, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(128, 256, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(128)
        self.bn2 = nn.BatchNorm1d(256)
        self.global_pool = nn.AdaptiveMaxPool1d(1)
        self.dropout = nn.Dropout(0.4)
        self.fc1 = nn.Linear(256, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = x.unsqueeze(1)
        x = F.leaky_relu(self.bn1(self.conv1(x)))
        x = F.leaky_relu(self.bn2(self.conv2(x)))
        x = self.global_pool(x).squeeze(-1)
        x = self.dropout(F.leaky_relu(self.fc1(x)))
        return self.fc2(x)

model = FullFeatureCNN(input_dim=X_train.shape[1], num_classes=len(df_train["label_enc"].unique())).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)
epochs = 10

for epoch in range(epochs):
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    # Validation
    model.eval()
    val_loss, y_true, y_pred = 0, [], []
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            val_loss += criterion(outputs, y_batch).item()
            preds = torch.argmax(outputs, dim=1)
            y_pred.extend(preds.cpu().tolist())
            y_true.extend(y_batch.cpu().tolist())

    val_acc = accuracy_score(y_true, y_pred)
    print(f"Epoch {epoch+1}/{epochs} | Train Loss: {total_loss/len(train_loader):.4f} | "
          f"Val Loss: {val_loss/len(val_loader):.4f} | Val Acc: {val_acc:.4f}")


Epoch 1/10 | Train Loss: 1.8264 | Val Loss: 1.7742 | Val Acc: 0.1604
Epoch 2/10 | Train Loss: 1.7687 | Val Loss: 1.7606 | Val Acc: 0.2040
Epoch 3/10 | Train Loss: 1.7618 | Val Loss: 1.7608 | Val Acc: 0.2002
Epoch 4/10 | Train Loss: 1.7591 | Val Loss: 1.7614 | Val Acc: 0.1931
